# Unitree G1 — PPO/AIRL train dài trên Kaggle
Clone branch `feature/kaggle-airl-16k`; Dataset `nvhiep2409/airl-ppo`; Internet + một T4.
Mặc định tiếp tục từ COMMON PPO999, **589824000 transitions/arm** (1500 updates tại 16384 env),
chunk 23592960 transitions (60 updates). Đây là continuation, không phải fresh training.
PPO cập nhật actor/critic; AIRL cập nhật actor/critic và g/h. Simulator reset ở mỗi chunk.

Lần đầu chạy theo thứ tự. Có `selected.json` đã đo thì đặt `SELECTED_INPUT` để bỏ sweep.
Session sau: attach **toàn bộ outputs cũ**, đổi SESSION, đặt RESUME_STATE trỏ tới
`<SESSION_cũ>_paired/paired_state.json`; notebook pin commit và giữ budget/seed/selection.
State tự chọn arm còn thiếu; không dùng checkpoint AIRL cho PPO.


In [ ]:
import ctypes.util
import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/nvhiep249/mjlab.git"
REPO_BRANCH = "feature/kaggle-airl-16k"
REPO_COMMIT = None  # Set a full commit SHA to replay a saved run exactly.
INPUT_ROOT = Path("/kaggle/input/datasets/nvhiep2409/airl-ppo")
OUTPUT = Path("/kaggle/working/outputs")
GPU = 0
SESSION = "long01"  # New SESSION when importing previous outputs into a new runtime.
SOURCE = Path("/tmp") / f"mjlab-github-{SESSION}"
SEED = 42
TRANSITIONS = 589824000  # Per arm: 1500 additional updates at 16384 envs.
CHUNK_TRANSITIONS = 23592960  # 60 updates/chunk at 16384 envs.
SESSION_SECONDS = 28800  # Chosen training budget, not a Kaggle platform limit.
RESUME_STATE = None  # Path('/kaggle/input/.../long01_paired/paired_state.json')
SELECTED_INPUT = None  # Optional existing selected.json from this runtime/dataset.
PAIR = OUTPUT / f"{SESSION}_paired"
resume_path = (
  PAIR / "paired_state.json" if (PAIR / "paired_state.json").is_file() else RESUME_STATE
)
if resume_path is not None:
  resume_path = Path(resume_path)
  saved_state = json.loads(resume_path.read_text())
  REPO_COMMIT = saved_state["source_commit"]
  SEED = saved_state["config"]["seed"]
  TRANSITIONS = saved_state["config"]["target_transitions"]
  CHUNK_TRANSITIONS = saved_state["config"]["chunk_transitions"]
  SELECTED_INPUT = resume_path.parent / "selected.json"
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"
os.environ["PYTHONUTF8"] = "1"
os.environ["MUJOCO_GL"] = "egl"
OUTPUT.mkdir(parents=True, exist_ok=True)


def find_input(names):
  for name in names:
    matches = list(INPUT_ROOT.rglob(name))
    if len(matches) > 1:
      raise RuntimeError(f"Ambiguous input {name}: {matches}; set the path explicitly")
    if matches:
      return matches[0]
  raise FileNotFoundError(
    f"Attach nvhiep2409/airl-ppo; missing {names} under {INPUT_ROOT}"
  )


EXPERT = find_input(("airl_expert_1p2_deterministic_v1.pt",))
COMMON = find_input(("ppo_common_999.pt", "model_999.pt"))
INPUT = EXPERT.parent
print("Expert:", EXPERT, "Common PPO:", COMMON)
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
def digest(path):
  hasher = hashlib.sha256()
  with path.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
      hasher.update(chunk)
  return hasher.hexdigest()


DATA_FILES = {
  "expert": {
    "sha256": "3cf8b910c4957fc80d257762d55945e5777411ba2785659dfb29dd08d69e72e8",
    "bytes": 382726465,
  },
  "common_ppo": {
    "sha256": "b318a6da5db37a6795e4a943d0d52eaa75743bd403341ecf47b10b3f20972dc8",
    "bytes": 5316259,
  },
}
for name, path in (("expert", EXPERT), ("common_ppo", COMMON)):
  expected = DATA_FILES[name]
  if path.stat().st_size != expected["bytes"] or digest(path) != expected["sha256"]:
    raise RuntimeError(
      f"Wrong qualified input: {name}, {path}; do not train on a different file"
    )
(OUTPUT / f"{SESSION}_data_manifest.json").write_text(json.dumps(DATA_FILES, indent=2))
print("Expert and common PPO SHA256 verified")

In [ ]:
if SOURCE.exists():
  raise FileExistsError(
    f"Use a new SESSION or a fresh runtime; source already exists: {SOURCE}"
  )
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
subprocess.run(
  [
    "git",
    "clone",
    "--config",
    "core.autocrlf=false",
    "--depth",
    "1",
    "--single-branch",
    "--branch",
    REPO_BRANCH,
    REPO_URL,
    str(SOURCE),
  ],
  check=True,
  env=git_env,
)
if REPO_COMMIT is not None:
  subprocess.run(
    ["git", "fetch", "--depth", "1", "origin", REPO_COMMIT],
    cwd=SOURCE,
    check=True,
    env=git_env,
  )
  subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=SOURCE, check=True)
source_commit = subprocess.check_output(
  ["git", "rev-parse", "HEAD"], cwd=SOURCE, text=True
).strip()
if REPO_COMMIT is not None and source_commit != REPO_COMMIT:
  raise RuntimeError("Requested source commit was not checked out")
manifest = json.loads(
  (SOURCE / "scripts/cloud/kaggle_source_manifest.json").read_text()
)
for relative, expected in manifest["source_files"].items():
  target = (SOURCE / relative).resolve()
  if not target.is_relative_to(SOURCE.resolve()) or digest(target) != expected:
    raise RuntimeError(f"Source hash mismatch: {relative}")
provenance = {
  "repo": REPO_URL,
  "branch": REPO_BRANCH,
  "commit": source_commit,
  "source_files": manifest["source_files"],
  "data_files": DATA_FILES,
}
(OUTPUT / f"{SESSION}_github_source.json").write_text(json.dumps(provenance, indent=2))
print("Verified GitHub source:", source_commit)

In [ ]:
if not shutil.which("uv"):
  subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
if not ctypes.util.find_library("EGL"):
  subprocess.run(["apt-get", "update", "-qq"], check=True)
  subprocess.run(["apt-get", "install", "-y", "libegl1"], check=True)
subprocess.run(
  ["uv", "sync", "--locked", "--no-dev", "--extra", "cu128", "--python", "3.11"],
  cwd=SOURCE,
  check=True,
)
subprocess.run(
  [
    "uv",
    "run",
    "--no-sync",
    "python",
    "-c",
    "import importlib.metadata as m; print({p:m.version(p) for p in "
    "['torch','warp-lang','mujoco','mujoco-warp','rsl-rl-lib']})",
  ],
  cwd=SOURCE,
  check=True,
)

TOOL = "scripts/cloud/kaggle_airl.py"


def cloud(*arguments):
  return subprocess.run(
    ["uv", "run", "--no-sync", "python", TOOL, *map(str, arguments)],
    cwd=SOURCE,
    check=True,
  )

## Smoke lần đầu; resume bỏ qua smoke và baseline
5 updates chỉ kiểm tra integration. Không dùng smoke để đánh giá D.

In [ ]:
if resume_path is None:
  for arm in ("ppo", "airl"):
    target = OUTPUT / f"{SESSION}_smoke_{arm}"
    cloud(
      "worker",
      "--input",
      INPUT,
      "--checkpoint",
      COMMON,
      "--output",
      target,
      "--result",
      target / "run_summary.json",
      "--envs",
      1024,
      "--minibatches",
      4,
      "--updates",
      5,
      "--warmup",
      0,
      "--gpu",
      GPU,
      "--arm",
      arm,
    )

  smoke = json.loads((OUTPUT / f"{SESSION}_smoke_airl/run_summary.json").read_text())
  checkpoint = max(
    Path(smoke["run_dir"]).glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[-1])
  )
  target = OUTPUT / f"{SESSION}_resume_smoke"
  cloud(
    "worker",
    "--input",
    INPUT,
    "--checkpoint",
    checkpoint,
    "--resume",
    "--output",
    target,
    "--result",
    target / "run_summary.json",
    "--envs",
    1024,
    "--minibatches",
    4,
    "--updates",
    1,
    "--warmup",
    0,
    "--gpu",
    GPU,
    "--arm",
    "airl",
  )

In [ ]:
def evaluate(checkpoint, output, seed, role="endpoint"):
  subprocess.run(
    [
      "uv",
      "run",
      "--no-sync",
      "gail-evaluate-expert",
      "--task",
      manifest["task"],
      "--checkpoint-file",
      str(checkpoint),
      "--output-file",
      str(output),
      "--evaluation-role",
      role,
      "--device",
      "cuda:0",
      "--episodes-per-command",
      "100",
      "--seed",
      str(seed),
      "--lin-vel-x",
      "1.2",
      "--lin-vel-y",
      "0.0",
      "--ang-vel-z",
      "0.0",
    ],
    cwd=SOURCE,
    check=True,
    env={**os.environ, "CUDA_VISIBLE_DEVICES": str(GPU)},
  )
  return json.loads(output.read_text())


if resume_path is None:
  baseline = evaluate(
    COMMON, OUTPUT / f"{SESSION}_common_base_seed42005.json", 42005, role="baseline"
  )
  print("Common PPO baseline gate:", baseline["gate"])
  if not baseline["gate"]["passed"]:
    print(
      "Baseline chưa đạt task gate. Expert dataset được qualify riêng; "
      "giữ report này và tiếp tục sweep + paired pilot nếu metrics hợp lệ."
    )

## Chọn cấu hình bằng sweep hoặc dùng selection đã đo
Giữ 24 steps/env, 5 epochs, 1 D update với batch1024. Resume dùng selection đã lưu.

In [ ]:
if SELECTED_INPUT is None:
  SWEEP = OUTPUT / f"{SESSION}_sweep"
  cloud(
    "sweep",
    "--input",
    INPUT,
    "--checkpoint",
    COMMON,
    "--output",
    SWEEP,
    "--arm",
    "airl",
    "--gpu",
    GPU,
    "--envs",
    16384,
    20480,
    24576,
    32768,
    "--minibatches",
    4,
    8,
    "--warmup",
    3,
    "--measure",
    8,
  )
else:
  SWEEP = Path(SELECTED_INPUT).parent
SELECTED = SWEEP / "selected.json" if SELECTED_INPUT is None else Path(SELECTED_INPUT)
selected = json.loads(SELECTED.read_text())
rollout = selected["envs"] * 24
assert selected["ok"] and selected["envs"] > 16000
assert TRANSITIONS % rollout == 0 and CHUNK_TRANSITIONS % rollout == 0
print(json.dumps(selected, indent=2))
print("Additional updates/arm:", TRANSITIONS // rollout)
print("Updates/chunk:", CHUNK_TRANSITIONS // rollout)
print(
  "AIRL estimated training hours (excluding startup/checkpoints):",
  TRANSITIONS // rollout * selected["median_update_seconds"] / 3600,
)

## Train dài theo chunk
Chỉ tăng counters sau chunk hoàn tất và checkpoint hash được xác minh. Rerun cell này giữ tiến độ.
Nếu session budget hết: `complete=False` là trạng thái tạm dừng, không phải hội tụ.
Nếu worker lỗi: giữ state, sửa nguyên nhân rồi rerun; chunk chưa commit chạy lại từ checkpoint trước.
Không nhập state cũ vào output đang có state khác. Khi session mới, giữ cả thư mục paired
(initial.pt, selected.json, paired_state.json, checkpoint PPO/AIRL); không chỉ lấy JSON.
Torch RNG được khôi phục cho cả hai arm; simulator/Python/NumPy/Warp state không được restore chính xác.


In [ ]:
arguments = [
  "paired",
  "--input",
  INPUT,
  "--checkpoint",
  COMMON,
  "--selected",
  SELECTED,
  "--output",
  PAIR,
  "--transitions",
  TRANSITIONS,
  "--chunk-transitions",
  CHUNK_TRANSITIONS,
  "--session-seconds",
  SESSION_SECONDS,
  "--gpu",
  GPU,
  "--seed",
  SEED,
]
if (PAIR / "paired_state.json").is_file():
  arguments[arguments.index("--selected") + 1] = PAIR / "selected.json"
elif RESUME_STATE is not None:
  arguments += ["--state", Path(RESUME_STATE)]
cloud(*arguments)
state = json.loads((PAIR / "paired_state.json").read_text())
print("Full budget complete:", state["complete"])
print(
  "Committed additional updates:",
  {
    arm: row["transitions"] // (state["config"]["envs"] * 24)
    for arm, row in state["arms"].items()
  },
)

## Endpoint cùng budget
Success≥95%, linear≤0.25, yaw≤0.20, upright≥0.97; không hạ gate. Partial endpoint được in rõ budget.

In [ ]:
state = json.loads((PAIR / "paired_state.json").read_text())
counts = [state["arms"][arm]["transitions"] for arm in ("ppo", "airl")]
if counts[0] == counts[1] and counts[0] > 0:
  print("Endpoint budget/arm:", counts[0], "Full budget complete:", state["complete"])
  for arm in ("ppo", "airl"):
    row = state["arms"][arm]
    checkpoint = PAIR / row["checkpoint"]
    assert digest(checkpoint) == row["checkpoint_sha256"]
    for seed in (42005, 42006):
      evaluate(
        checkpoint, OUTPUT / f"{SESSION}_{arm}_t{counts[0]}_eval_seed{seed}.json", seed
      )
else:
  print("Skip paired endpoint: unequal/zero committed budgets; resume training first.")

## Export g/h để frozen
Export không chứng minh reward donor đã tốt. Giữ endpoint và so PPO cùng budget.
Artifact giữ g/h, gamma, normalization. Khi tích hợp frozen: `r_total = r_env + 0.01*f`,
`f = g(s,c) + gamma*(1-terminated)*h(s_next,c_next) - h(s,c)`; D updates=0.
Xem README để nạp artifact và hook reward; frozen PPO launcher chưa được tích hợp.


In [ ]:
state = json.loads((PAIR / "paired_state.json").read_text())
if state["complete"]:
  row = state["arms"]["airl"]
  checkpoint = PAIR / row["checkpoint"]
  assert digest(checkpoint) == row["checkpoint_sha256"]
  frozen_file = OUTPUT / f"{SESSION}_frozen_airl_{row['checkpoint_sha256'][:12]}.pt"
  if frozen_file.exists():
    raise FileExistsError(
      f"Preserve exported artifact; choose a new output: {frozen_file}"
    )
  subprocess.run(
    [
      "uv",
      "run",
      "--no-sync",
      "python",
      "-m",
      "mjlab.scripts.airl_export_reward",
      "--checkpoint-file",
      str(checkpoint),
      "--output-file",
      str(frozen_file),
    ],
    cwd=SOURCE,
    check=True,
  )
  print("Frozen reward:", frozen_file, "SHA256:", digest(frozen_file))
else:
  print("Full budget pending; save outputs and resume before the final export.")

In [ ]:
hashes = {
  path.relative_to(OUTPUT).as_posix(): digest(path)
  for path in OUTPUT.rglob("*")
  if path.is_file() and path.name != "output_hashes.json"
}
(OUTPUT / "output_hashes.json").write_text(json.dumps(hashes, indent=2))
print("Save notebook version with outputs, then download /kaggle/working/outputs")